# 04｜更難的無答案題：題目不變，把正確段落拿掉

**為什麼要做：** 03 的無答案題是「Collection B 的題目拿去問 A」，主題完全不重疊，撈到的段落一看就無關，模型很容易判斷要拒答。
實際使用時更常見的情況是：**文件裡有相關內容，但剛好沒寫到這件事。**

**做法：** 題目留在原本的知識庫，只在檢索時把它的正確段落排除（`exclude_ids`），其他段落都還在。

| 來源 | 拿掉什麼 | 模型看到什麼 | 該做什麼 |
|---|---|---|---|
| SQuAD | 唯一的正確段落 | 同一篇文章、主題相近的其他段落 | 拒答 |
| HotpotQA 橋接題 | 含答案的那一段（第二跳） | 第一跳的段落還可能在，知道中間的實體，但沒有最後的答案 | 拒答，不要用記憶補 |
| HotpotQA 比較題（yes/no） | 隨機拿掉一邊 | 只有其中一個實體的資料 | 拒答，不要猜 yes/no |

題目由 `python -m tools.datasets.make_hard_noanswer` 產生（規則寫在腳本開頭）：如果原答案的字串還出現在知識庫其他段落，那題就跳過，因為可能其實答得出來。
dev 的題目只從 03 跑過完整問答的前 150 題裡挑，所以每題都能和「正確段落還在時」的結果配對。

**兩種模式：**
- `rag`：正常檢索，但排除被拿掉的段落。
- `partial_oracle`（只有 HotpotQA）：Context **只放剩下的那段正確段落**。這是最乾淨的測試：模型手上只有一半證據，看它會不會硬答。

**回答的分類（自動，之後要人工確認）：**
- `abstain`：拒答 → 正確行為
- `orig_answer`：還是答出原本的答案 → 可能是模型背過、從剩下的段落合理推出來，或答案換了寫法留在別的段落（字串檢查抓不到）
- `other_answer`：答了別的東西 → 多半是幻覺

**執行時間：** dev 約 100 題 `rag` ＋ 50 題 `partial_oracle`，4B 約 5～10 分鐘。

In [17]:
import json
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "collections").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src import rag_core as rc

SPLIT = "dev"                  # "dev" | "holdout"（最後驗收才跑）
RETRIEVER = "dense"            # "dense" | "hybrid"
RRF_K = 60
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
GEN_MODEL, GEN_THINK = "qwen3:4b-instruct", None
OLLAMA_URL = "http://localhost:11434"
NUM_CTX = 4096
CONTEXT_CHAR_BUDGET = 8000
TOP_K = 3
RUN_E2E = True                 # False：只做第 2 節（不呼叫 LLM）
RUN_PARTIAL_ORACLE = True      # HotpotQA 另外跑「只給剩下那段正確段落」
CORRECT_F1 = 0.5

LOAD_RUN = None                # 填既有 run_id 就只讀紀錄畫表

SPLIT="holdout"

pd.set_option("display.max_colwidth", 90)
print("ROOT =", ROOT)

ROOT = <專案根目錄>


## 1. 載入題目與知識庫

In [18]:
# 正確段落還在時的結果（03 的 run），用來配對比較
BASELINE_RUNS = {
    ("dense", "dev"): "20261026_001846_bench_dense",
    ("hybrid", "dev"): "20261026_155225_bench_hybrid",
    ("dense", "holdout"): "20261026_185546_bench_dense_holdout",
    ("hybrid", "holdout"): "20261026_191449_bench_hybrid_holdout",
}
EASY_NOANSWER = {"dev": "squad_b_noanswer_dev50.jsonl", "holdout": "squad_b_noanswer_holdout50.jsonl"}[SPLIT]

qfile = ROOT / "eval" / f"hard_noanswer_{SPLIT}.jsonl"
if not qfile.exists():
    raise FileNotFoundError(f"找不到 {qfile.name}，先執行：python -m tools.datasets.make_hard_noanswer")
questions = rc.load_jsonl(qfile)
assert all(q["split"] == f"hard_{SPLIT}" for q in questions), "題目檔和 SPLIT 不符"

data = {}
for coll in sorted({q["collection"] for q in questions}):
    docs = rc.load_collection(ROOT, coll)
    if not docs:
        print(f"[略過] 找不到知識庫 {coll}")
        continue
    data[coll] = {"docs": docs, "doc_by_id": {d["doc_id"]: d for d in docs}}
    bad = [q["qid"] for q in questions if q["collection"] == coll
           and not set(q["removed_doc_ids"] + q["kept_gold_ids"]) <= set(data[coll]["doc_by_id"])]
    assert not bad, f"{coll}: {len(bad)} 題的段落不在知識庫裡（知識庫重建過？）"
questions = [q for q in questions if q["collection"] in data]

print(f"SPLIT = {SPLIT}　RETRIEVER = {RETRIEVER}　題目 {len(questions)} 題")
display(pd.DataFrame(questions).groupby(["source", "type"]).size().rename("題數").to_frame())

SPLIT = holdout　RETRIEVER = dense　題目 91 題


題數
source type          
hotpot bridge      35
       comparison   6
squad  single      50

In [19]:
embedder = rc.Embedder(EMBED_MODEL)
for coll, d in data.items():
    idx = rc.build_or_load_index(ROOT, d["docs"], coll, embedder)
    dense = rc.DenseRetriever(idx, d["docs"], embedder)
    d["dense"] = dense
    d["retriever"] = dense if RETRIEVER == "dense" else rc.HybridRetriever(dense, rc.BM25Retriever(d["docs"], coll), rrf_k=RRF_K)
    print(f"{coll}: {d['retriever'].index_key}")

hotpot_dev400: hotpot_dev400__BAAI-bge-small-en-v1-5__7a77db1d36b0
squad_v11_dev500: squad_v11_dev500__BAAI-bge-small-en-v1-5__153e36d1e3b4


## 2. 不呼叫 LLM：拿掉正確段落後，撈到的是什麼？

用 dense 的相似度分數（cosine）比較三組：
- **可回答**：同一批題目，正確段落還在時的第 1 名分數
- **簡單無答案**：03 的 B 問 A
- **困難無答案**：這次，正確段落被排除後的第 1 名分數

如果困難組的分數和「可回答」差不多，就代表**光看檢索分數分不出來**，模型必須真的讀內容才能判斷要不要拒答。

另外看撈到的段落和原題有多接近：SQuAD 看 Top-k 裡有幾段來自**同一篇文章**；HotpotQA 看**剩下那段正確段落**有沒有被撈到。

In [20]:
rows = []
for q in questions:
    d = data[q["collection"]]
    with_gold = d["dense"].search(q["question"], TOP_K)
    hits = d["retriever"].search(q["question"], TOP_K, exclude=q["removed_doc_ids"])
    hits_dense = d["dense"].search(q["question"], 1, exclude=q["removed_doc_ids"])
    removed_titles = {d["doc_by_id"][x]["title"] for x in q["removed_doc_ids"]}
    rows.append({"qid": q["qid"], "source": q["source"], "type": q["type"],
                 "top1_with_gold": with_gold[0]["score"], "top1_hard": hits_dense[0]["score"],
                 "same_article_in_topk": sum(h["title"] in removed_titles for h in hits),
                 "kept_gold_in_topk": sum(h["doc_id"] in q["kept_gold_ids"] for h in hits),
                 "topk_titles": [h["title"] for h in hits]})
ret = pd.DataFrame(rows)

easy_scores = []
if "squad_v11_dev500" in data and (ROOT / "eval" / EASY_NOANSWER).exists():
    easy = rc.load_jsonl(ROOT / "eval" / EASY_NOANSWER)
    easy_scores = [data["squad_v11_dev500"]["dense"].search(q["question"], 1)[0]["score"] for q in easy]

groups = {}
for src, g in ret.groupby("source"):
    groups[f"{src}：可回答（正確段落還在）"] = g.top1_with_gold
    groups[f"{src}：困難無答案"] = g.top1_hard
if easy_scores:
    groups["squad：簡單無答案（B 問 A）"] = pd.Series(easy_scores)
summary = pd.DataFrame({k: v.describe(percentiles=[.1, .5, .9])[["count", "10%", "50%", "90%"]] for k, v in groups.items()}).T
print("dense 第 1 名的相似度分數")
display(summary.round(3))

if "squad" in set(ret.source):
    s = ret[ret.source == "squad"]
    print(f"SQuAD：Top-{TOP_K} 平均有 {s.same_article_in_topk.mean():.2f} 段來自同一篇文章；"
          f"{(s.same_article_in_topk > 0).mean():.0%} 的題目至少有一段")
if "hotpot" in set(ret.source):
    h = ret[ret.source == "hotpot"]
    print(f"HotpotQA：剩下那段正確段落在 Top-{TOP_K} 裡的比例 {h.kept_gold_in_topk.mean():.0%}")

dense 第 1 名的相似度分數


,count,10%,50%,90%
hotpot：可回答（正確段落還在）,41.0,0.719,0.755,0.814
hotpot：困難無答案,41.0,0.691,0.749,0.804
squad：可回答（正確段落還在）,50.0,0.604,0.727,0.824
squad：困難無答案,50.0,0.586,0.671,0.770
squad：簡單無答案（B 問 A）,50.0,0.524,0.565,0.624


SQuAD：Top-3 平均有 2.68 段來自同一篇文章；98% 的題目至少有一段
HotpotQA：剩下那段正確段落在 Top-3 裡的比例 98%


## 3. 完整問答

In [21]:
def run_e2e():
    gen = rc.OllamaGenerator(GEN_MODEL, OLLAMA_URL, NUM_CTX, GEN_THINK)
    gen.check()
    jobs = [(q, "rag") for q in questions]
    if RUN_PARTIAL_ORACLE:
        jobs += [(q, "partial_oracle") for q in questions if q["kept_gold_ids"]]
    recs, t0 = [], time.time()
    for i, (q, mode) in enumerate(jobs, 1):
        d = data[q["collection"]]
        gold_docs = [d["doc_by_id"][g] for g in q["kept_gold_ids"]] if mode == "partial_oracle" else None
        r = rc.ask(q, mode, gen, d["retriever"], k=TOP_K, char_budget=CONTEXT_CHAR_BUDGET,
                   gold_docs=gold_docs, exclude_ids=q["removed_doc_ids"])
        s = rc.score(r["answer"], q["orig_answers"], q["question"])     # 和「原本的答案」比
        r.update({f"orig_{k}": v for k, v in s.items()})
        r.update(source=q["source"], type=q["type"], retriever=RETRIEVER, orig_answers=q["orig_answers"],
                 removed_doc_ids=q["removed_doc_ids"], kept_gold_ids=q["kept_gold_ids"])
        recs.append(r)
        if i % 25 == 0 or i == len(jobs):
            print(f"{i}/{len(jobs)}，{time.time() - t0:.0f}s")
    return recs


if LOAD_RUN is None:
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S") + f"_hardnoans_{RETRIEVER}_{SPLIT}"
    run_dir = ROOT / "results" / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(json.dumps({
        "run_id": run_id, "split": SPLIT, "questions": qfile.name, "retriever": RETRIEVER, "rrf_k": RRF_K,
        "embed_model": EMBED_MODEL, "gen_model": GEN_MODEL, "gen_think": GEN_THINK, "top_k": TOP_K,
        "num_ctx": NUM_CTX, "context_char_budget": CONTEXT_CHAR_BUDGET, "run_partial_oracle": RUN_PARTIAL_ORACLE,
        "index_keys": {c: d["retriever"].index_key for c, d in data.items()}, "system_rag": rc.SYSTEM_RAG},
        indent=2, ensure_ascii=False), encoding="utf-8")
    ret.to_json(run_dir / "retrieval.jsonl", orient="records", lines=True, force_ascii=False)
    records = run_e2e() if RUN_E2E else []
    if records:
        with (run_dir / "records.jsonl").open("w", encoding="utf-8") as f:
            for r in records:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("已存到", run_dir)
else:
    run_id, run_dir = LOAD_RUN, ROOT / "results" / LOAD_RUN
    rec_file = run_dir / "records.jsonl"
    records = rc.load_jsonl(rec_file) if rec_file.exists() else []
    print(f"讀取 {run_id}：{len(records)} 筆")

25/132，62s
50/132，123s
75/132，185s
100/132，245s
125/132，302s
132/132，318s
已存到 <專案根目錄>\results\20261026_230800_hardnoans_dense_holdout


## 4. 結果

拒答率越高越好。`orig_answer` 和 `other_answer` 都算失敗，但原因不同，第 6 節要人工看。

In [22]:
df = pd.DataFrame(records)


def label(r):
    if r["abstained"]:
        return "abstain"
    if r["orig_contains_gold"] or (r["orig_f1_qstrip"] or 0) >= CORRECT_F1:
        return "orig_answer"
    return "other_answer"


if df.empty:
    print("沒有完整問答的紀錄（RUN_E2E = False）")
else:
    df["label"] = df.apply(label, axis=1)
    tab = pd.crosstab([df["mode"], df.source, df.type], df.label, margins=True, margins_name="合計")
    tab = tab.reindex(columns=[c for c in ["abstain", "orig_answer", "other_answer", "合計"] if c in tab.columns])
    tab["拒答率"] = (tab.get("abstain", 0) / tab["合計"]).round(3)
    display(tab)

    base_id = BASELINE_RUNS.get((RETRIEVER, SPLIT))
    base_file = ROOT / "results" / str(base_id) / "records.jsonl"
    if base_file.exists():
        base = pd.DataFrame(rc.load_jsonl(base_file))
        easy = base[base.dataset == "noanswer"]
        if len(easy):
            print(f"對照：簡單無答案（B 問 A，{base_id}）拒答 {int(easy.abstained.sum())}/{len(easy)}")

label                             abstain  orig_answer  other_answer   合計  \
mode           source type                                                  
partial_oracle hotpot bridge           24            5             6   35   
                      comparison        6            0             0    6   
rag            hotpot bridge           26            3             6   35   
                      comparison        4            2             0    6   
               squad  single           39            1            10   50   
合計                                     99           11            22  132   

label                               拒答率  
mode           source type               
partial_oracle hotpot bridge      0.686  
                      comparison  1.000  
rag            hotpot bridge      0.743  
                      comparison  0.667  
               squad  single      0.780  
合計                                0.750

對照：簡單無答案（B 問 A，20261026_185546_bench_dense_holdout）拒答 49/50


## 5. 配對：同一題，正確段落在 vs. 不在

列：正確段落還在時（03 的結果）有沒有答對；欄：拿掉之後的反應。
最理想的是「有證據就答對、沒證據就拒答」。如果原本就答錯的題目，拿掉證據後反而「答出原答案」，那多半是模型背過的。

In [23]:
if not df.empty and base_file.exists():
    b = base[(base["mode"] == "rag") & base.dataset.isin(["squad", "hotpot"])].copy()
    b["有證據時答對"] = b.contains_gold.astype(bool) | (b.f1_qstrip.fillna(0) >= CORRECT_F1)
    b = b.rename(columns={"dataset": "source", "answer": "answer_with_gold"})[["source", "qid", "有證據時答對", "answer_with_gold"]]
    paired = df[df["mode"] == "rag"].merge(b, on=["source", "qid"], how="left")
    print(f"配對到 {paired['有證據時答對'].notna().sum()}/{len(paired)} 題（基準 {base_id}）")
    display(pd.crosstab([paired.source, paired["有證據時答對"]], paired.label, margins=True, margins_name="合計"))
else:
    paired = pd.DataFrame()
    print("找不到基準 run，略過配對")

配對到 91/91 題（基準 20261026_185546_bench_dense_holdout）


label          abstain  orig_answer  other_answer  合計
source 有證據時答對                                        
hotpot False        14            0             2  16
       True         16            5             4  25
squad  False         9            0             2  11
       True         30            1             8  39
合計                  69            6            16  91

## 6. 沒有拒答的題目 → 人工核對

輸出到 `reports/hard_noanswer_<run_id>_review.csv`，請填 `human_label`：

| human_label | 意思 |
|---|---|
| `hallucination` | 段落沒有支持這個答案，模型自己編的或猜的 |
| `parametric` | 答出原本的正確答案，但段落沒寫 → 靠預訓練記憶 |
| `supported_inference` | 剩下的段落其實能推出答案 → **這題不算真的無答案**，要從題庫剔除 |
| `alias_leak` | 剩下的段落用別的寫法寫了答案（例如 Ed Roland 對 Edgar Eugene Roland, Jr.）→ 同上 |

後兩種是**題目的問題**，不是模型的問題，統計拒答率時要扣掉。

In [24]:
if not df.empty:
    fail = df[~df.abstained].copy()
    if not paired.empty:
        fail = fail.merge(paired[["source", "qid", "有證據時答對", "answer_with_gold"]], on=["source", "qid"], how="left")
    by_coll = {q["qid"]: q["collection"] for q in questions}
    fail["context"] = [
        "\n\n".join(f"[{c}] {data[by_coll[qid]]['doc_by_id'][c]['text']}" for c in ctx)
        for qid, ctx in zip(fail.qid, fail.context_ids)]
    cols = ["qid", "source", "type", "mode", "question", "orig_answers", "answer", "label", "cited_ids",
            "有證據時答對", "answer_with_gold", "removed_doc_ids", "context"]
    out = fail[[c for c in cols if c in fail.columns]].assign(human_label="", note="")
    review = ROOT / "reports" / f"hard_noanswer_{run_id}_review.csv"
    review.parent.mkdir(exist_ok=True)
    out.to_csv(review, index=False, encoding="utf-8-sig")
    print(f"{len(out)} 題沒有拒答 → {review.relative_to(ROOT)}")
    display(out[["source", "mode", "question", "orig_answers", "answer", "label"]])

33 題沒有拒答 → reports\hard_noanswer_20261026_230800_hardnoans_dense_holdout_review.csv


,source,mode,question,orig_answers,answer,label
0,squad,rag,What was a Happy Days spinoff that debuted in the 1980s on ABC?,[Mork & Mindy],Laverne & Shirley,other_answer
1,squad,rag,What kind of experiments of rock deformation do structural geologists perform?,"[analog and numerical experiments, analog and numerical]","Experiments involving orogenic wedges, where horizontal sand layers are pulled into a ...",other_answer
2,squad,rag,What is the hypersensitive response of a plant immune system?,"[infected cells seal themselves off and undergo programmed cell death, programmed cell...",The hypersensitive response of a plant immune system is a defense mechanism against pa...,other_answer
3,squad,rag,Who are likely participants in creating an overall plan for the financial management o...,"[Mortgage bankers, accountants, and cost engineers]","Architect, construction manager, design engineer, construction engineer, project manager",other_answer
4,squad,rag,What proclamation officially ended limited Huguenot autonomy?,[Edict of Alès],Revocation of the Edict of Nantes,orig_answer
5,squad,rag,Where did these uprisings take place?,"[southwestern France, mainly in southwestern France]","Ballarat, Paris, Aix, Bordeaux, Bourges, Lyons, Meaux, Orleans, Rouen, Toulouse, Troyes",other_answer
6,squad,rag,What shape are granal thylakoids?,"[pancake-shaped circular disks, pancake, pancake-shaped]",flattened circular,other_answer
7,squad,rag,What other compensation did Tesla get from Westinghouse?,"[rent at the Hotel New Yorker, paying his rent, paying his rent at the Hotel New Yorker]","$2,000 per month as a consultant.",other_answer
8,squad,rag,What is the present-day location of this church?,[Soho Square],Trinity-St. Paul's Episcopal Church,other_answer
9,squad,rag,What kind of programs can be credited for ABC's success in the late 1950s?,[westerns and detective series],"Suspenseful dramas like Sherlock Holmes, Gang Busters, and Counterspy, and youth-orien...",other_answer


## 7. 怎麼解讀

- **困難組的拒答率明顯低於簡單組** → 03 的 50/50 是因為題目太簡單，不代表模型真的會判斷證據夠不夠。
- **`partial_oracle` 的拒答率低於 `rag`** → 給的段落越相關，模型越容易「順著補完」；這正是實際使用時最危險的情況。
- **`orig_answer` 很多，而且集中在「有證據時答錯」的題目** → 模型背過答案，No-RAG 的污染檢查應該一併看。
- 人工核對後，`supported_inference` / `alias_leak` 的題目要從題庫剔除，再重算拒答率。